# Using Machine Learning techniques to predict a stroke

## Imports

In [214]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

## Load in data, basic data information

In [215]:
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")

file_path = os.path.join(path, "healthcare-dataset-stroke-data.csv")
df = pd.read_csv(file_path)

df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [216]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


## Data cleaning

In [217]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='str')

### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [218]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

id values: [ 9046 51676 31112 ... 19723 37544 44679]
gender values: <StringArray>
['Male', 'Female', 'Other']
Length: 3, dtype: str
age values: [6.70e+01 6.10e+01 8.00e+01 4.90e+01 7.90e+01 8.10e+01 7.40e+01 6.90e+01
 5.90e+01 7.80e+01 5.40e+01 5.00e+01 6.40e+01 7.50e+01 6.00e+01 5.70e+01
 7.10e+01 5.20e+01 8.20e+01 6.50e+01 5.80e+01 4.20e+01 4.80e+01 7.20e+01
 6.30e+01 7.60e+01 3.90e+01 7.70e+01 7.30e+01 5.60e+01 4.50e+01 7.00e+01
 6.60e+01 5.10e+01 4.30e+01 6.80e+01 4.70e+01 5.30e+01 3.80e+01 5.50e+01
 1.32e+00 4.60e+01 3.20e+01 1.40e+01 3.00e+00 8.00e+00 3.70e+01 4.00e+01
 3.50e+01 2.00e+01 4.40e+01 2.50e+01 2.70e+01 2.30e+01 1.70e+01 1.30e+01
 4.00e+00 1.60e+01 2.20e+01 3.00e+01 2.90e+01 1.10e+01 2.10e+01 1.80e+01
 3.30e+01 2.40e+01 3.40e+01 3.60e+01 6.40e-01 4.10e+01 8.80e-01 5.00e+00
 2.60e+01 3.10e+01 7.00e+00 1.20e+01 6.20e+01 2.00e+00 9.00e+00 1.50e+01
 2.80e+01 1.00e+01 1.80e+00 3.20e-01 1.08e+00 1.90e+01 6.00e+00 1.16e+00
 1.00e+00 1.40e+00 1.72e+00 2.40e-01 1.64e+00 1.56e+0

### Adjusting column types

In [219]:
df["age"] = df["age"].astype("uint8")
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [220]:
df.isna().any().any()

np.True_

### Dealing with NaNs

BMI is the only columns with missing values. We fill them using the gender average.

In [221]:
df["bmi"] = df["bmi"].fillna(df.groupby("gender")["bmi"].transform("mean"))
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,Yes,Private,Urban,228.69,36.600000,formerly smoked,1
1,51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,29.065758,never smoked,1
2,31112,Male,80,0,1,Yes,Private,Rural,105.92,32.500000,never smoked,1
3,60182,Female,49,0,0,Yes,Private,Urban,171.23,34.400000,smokes,1
4,1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.000000,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80,1,0,Yes,Private,Urban,83.75,29.065758,never smoked,0
5106,44873,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.000000,never smoked,0
5107,19723,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.600000,never smoked,0
5108,37544,Male,51,0,0,Yes,Private,Rural,166.29,25.600000,formerly smoked,0


### Modifying the data

In [222]:
# Binary to 1/0
df["ever_married"] = df["ever_married"].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["gender", "work_type", "Residence_type", "smoking_status"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,9046,67,0,1,1,228.69,36.600000,1,0,1,...,0,1,0,0,0,1,0,1,0,0
1,51676,61,0,0,1,202.21,29.065758,1,1,0,...,0,0,1,0,1,0,0,0,1,0
2,31112,80,0,1,1,105.92,32.500000,1,0,1,...,0,1,0,0,1,0,0,0,1,0
3,60182,49,0,0,1,171.23,34.400000,1,1,0,...,0,1,0,0,0,1,0,0,0,1
4,1665,79,1,0,1,174.12,24.000000,1,1,0,...,0,0,1,0,1,0,0,0,1,0


In [223]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "stroke"] + ["stroke"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [224]:
def get_renamed_column(name):
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [225]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,gender_female,gender_male,gender_other,...,work_type_private,work_type_self-employed,work_type_children,residence_type_rural,residence_type_urban,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,stroke
0,9046,67,0,1,1,228.69,36.600000,0,1,0,...,1,0,0,0,1,0,1,0,0,1
1,51676,61,0,0,1,202.21,29.065758,1,0,0,...,0,1,0,1,0,0,0,1,0,1
2,31112,80,0,1,1,105.92,32.500000,0,1,0,...,1,0,0,1,0,0,0,1,0,1
3,60182,49,0,0,1,171.23,34.400000,1,0,0,...,1,0,0,0,1,0,0,0,1,1
4,1665,79,1,0,1,174.12,24.000000,1,0,0,...,0,1,0,1,0,0,0,1,0,1


### Check for duplicates

In [226]:
any(df.duplicated())

False

There are only unique records in the dataset already, we may proceed.

### Check for class imbalance

Ideally here should be a similar number of records in each class.

In [227]:
df[df.columns[-1]].sum() / df[df.columns[-1]].size

np.float64(0.0487279843444227)

Since stroke was confirmed only in 5% of patients we will use SMOTE to create synthetic data and deal with the imbalance.

## Preparing the train and test sets

### Train-test split

In [228]:
columns_to_drop = []
df = df.drop(columns=columns_to_drop)

In [229]:
X = df.drop(columns=[df.columns[-1]]) 
y = df[df.columns[-1]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Create models

### Random Forests

In [230]:
random_forest_models = {
    "RandomForest_100_5": RandomForestClassifier(
        n_estimators=100,
        max_depth=5
    ),
    "RandomForest_50_5": RandomForestClassifier(
        n_estimators=50,
        max_depth=5
    ),
    "RandomForest_50_10": RandomForestClassifier(
        n_estimators=50,
        max_depth=10
    ),
    "RandomForest_500_3": RandomForestClassifier(
        n_estimators=500,
        max_depth=3
    ),
    "RandomForest_200_10": RandomForestClassifier(
        n_estimators=200,
        max_depth=10
    ),
    "RandomForest_300_7": RandomForestClassifier(
        n_estimators=300,
        max_depth=7
    ),
}

### Neural Networks

In [231]:
input_shape = X_train.shape[1]

neural_networks_models = {
    "NN_64_32_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_32_32_32_1": Sequential([
        Dense(32, activation="sigmoid", input_shape=(input_shape,)),
        Dense(32, activation="sigmoid"),
        Dense(32, activation="sigmoid"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(64, activation="tanh"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_128_64_64_1": Sequential([
        Dense(128, activation="relu", input_shape=(input_shape,)),
        Dense(64, activation="relu"),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_32_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(32, activation="tanh"),
        Dense(64, activation="tanh"),
        Dense(1, activation="sigmoid")
    ]),
}

c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Train models

### Using SMOTE to deal with class imbalance

#### Cross-validation pipeline

In [232]:
def validate_models_smote(models, X, y):
    results = {}

    for name, model in models.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("smote", SMOTE(sampling_strategy="minority")),
            ("model", model)
        ])

        scores = cross_validate(pipeline, X, y, cv=5, scoring=["precision", "recall", "f1"])
        results[name] = {
            "precision": scores["test_precision"].mean(),
            "recall": scores["test_recall"].mean(),
            "f1": scores["test_f1"].mean(),
        }

    return results

In [233]:
results = validate_models_smote(random_forest_models, X_train, y_train)
for name, result in results.items():
    print(f"{name}: {result}\n")

RandomForest_100_5: {'precision': np.float64(0.11280421870633614), 'recall': np.float64(0.7008534850640114), 'f1': np.float64(0.19429482932235378)}

RandomForest_50_5: {'precision': np.float64(0.1135619686757896), 'recall': np.float64(0.684352773826458), 'f1': np.float64(0.19468812702301216)}

RandomForest_50_10: {'precision': np.float64(0.12772165856036824), 'recall': np.float64(0.3524893314366999), 'f1': np.float64(0.18707544427246053)}

RandomForest_500_3: {'precision': np.float64(0.1157790763760352), 'recall': np.float64(0.7916073968705548), 'f1': np.float64(0.2019729990296141)}

RandomForest_200_10: {'precision': np.float64(0.124929861096673), 'recall': np.float64(0.3365576102418208), 'f1': np.float64(0.18156709836309337)}

RandomForest_300_7: {'precision': np.float64(0.125334066779431), 'recall': np.float64(0.5881934566145093), 'f1': np.float64(0.2063284823793093)}



#### Training pipeline

In [234]:
def train_models_smote(models, X, y):

    trained_models = {}

    for name, model in models.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("smote", SMOTE(sampling_strategy="minority")),
            ("model", model)
        ])

        trained_pipeline = pipeline.fit(X, y)
        trained_models[name] = trained_pipeline

    return trained_models


In [235]:
trained_models = train_models_smote(random_forest_models, X_train, y_train)

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}\n")

RandomForest_100_5 Accuracy: 0.7348336594911937
RandomForest_100_5 Precision: 0.15737704918032788
RandomForest_100_5 Recall: 0.7741935483870968

RandomForest_50_5 Accuracy: 0.7172211350293543
RandomForest_50_5 Precision: 0.15076923076923077
RandomForest_50_5 Recall: 0.7903225806451613

RandomForest_50_10 Accuracy: 0.8317025440313112
RandomForest_50_10 Precision: 0.17647058823529413
RandomForest_50_10 Recall: 0.4838709677419355

RandomForest_500_3 Accuracy: 0.7123287671232876
RandomForest_500_3 Precision: 0.15269461077844312
RandomForest_500_3 Recall: 0.8225806451612904

RandomForest_200_10 Accuracy: 0.8287671232876712
RandomForest_200_10 Precision: 0.15757575757575756
RandomForest_200_10 Recall: 0.41935483870967744

RandomForest_300_7 Accuracy: 0.7710371819960861
RandomForest_300_7 Precision: 0.16666666666666666
RandomForest_300_7 Recall: 0.6935483870967742



### Using Ensemble to deal with class imbalance

#### Cross-validation pipeline

#### Training pipeline

### Using class weight to deal with the imbalance

#### Training Neural Networks

In [240]:
def train_neural_networks(models, X, y):
    results = {}

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    for name, model in models.items():
        model_copy = clone_model(model)
        model_copy.set_weights(model.get_weights()) 
        
        model_copy.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["precision", "recall"])
        model_copy.fit(X_scaled, y, epochs=50, batch_size=32, verbose=0, class_weight={0: 1, 1: 19}, validation_split=0.2)

        results[name] = model_copy

    return results, scaler

In [243]:
trained_models, scaler = train_neural_networks(neural_networks_models, X_train, y_train)
X_test_scaled = scaler.transform(X_test)

for name, model in trained_models.items():
    y_prob = model.predict(X_test_scaled).flatten()
    for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        print(f"CURRENT THRESHOLD: {threshold}")
        y_pred = (y_prob > threshold).astype(int) 

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        print(f"{name} Accuracy: {accuracy}")
        print(f"{name} Precision: {precision}")
        print(f"{name} Recall: {recall}\n")

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
CURRENT THRESHOLD: 0.1
NN_64_32_1 Accuracy: 0.6800391389432485
NN_64_32_1 Precision: 0.136986301369863
NN_64_32_1 Recall: 0.8064516129032258

CURRENT THRESHOLD: 0.2
NN_64_32_1 Accuracy: 0.7309197651663405
NN_64_32_1 Precision: 0.14381270903010032
NN_64_32_1 Recall: 0.6935483870967742

CURRENT THRESHOLD: 0.3
NN_64_32_1 Accuracy: 0.7514677103718199
NN_64_32_1 Precision: 0.13358778625954199
NN_64_32_1 Recall: 0.5645161290322581

CURRENT THRESHOLD: 0.4
NN_64_32_1 Accuracy: 0.786692759295499
NN_64_32_1 Precision: 0.14545454545454545
NN_64_32_1 Recall: 0.5161290322580645

CURRENT THRESHOLD: 0.5
NN_64_32_1 Accuracy: 0.8131115459882583
NN_64_32_1 Precision: 0.14754098360655737
NN_64_32_1 Recall: 0.43548387096774194

CURRENT THRESHOLD: 0.6
NN_64_32_1 Accuracy: 0.8395303326810176
NN_64_32_1 Precision: 0.16
NN_64_32_1 Recall: 0.3870967741935484

CURRENT THRESHOLD: 0.7
NN_64_32_1 Accuracy: 0.8581213307240705
NN_64_32_1 Precision: 0.15126050420168066
NN_64_32_

c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
CURRENT THRESHOLD: 0.1
NN_64_64_1 Accuracy: 0.6643835616438356
NN_64_64_1 Precision: 0.1273209549071618
NN_64_64_1 Recall: 0.7741935483870968

CURRENT THRESHOLD: 0.2
NN_64_64_1 Accuracy: 0.7270058708414873
NN_64_64_1 Precision: 0.13712374581939799
NN_64_64_1 Recall: 0.6612903225806451

CURRENT THRESHOLD: 0.3
NN_64_64_1 Accuracy: 0.7661448140900196
NN_64_64_1 Precision: 0.14741035856573706
NN_64_64_1 Recall: 0.5967741935483871

CURRENT THRESHOLD: 0.4
NN_64_64_1 Accuracy: 0.7906066536203522
NN_64_64_1 Precision: 0.15454545454545454
NN_64_64_1 Recall: 0.5483870967741935

CURRENT THRESHOLD: 0.5
NN_64_64_1 Accuracy: 0.8091976516634051
NN_64_64_1 Precision: 0.15897435897435896
NN_64_64_1 Recall: 0.5

CURRENT THRESHOLD: 0.6
NN_64_64_1 Accuracy: 0.8346379647749511
NN_64_64_1 Precision: 0.17575757575757575
NN_64_64_1 Recall: 0.46774193548387094

CURRENT THRESHOLD: 0.7
NN_64_64_1 Accuracy: 0.8542074363992173
NN_64_64_1 Precision: 0.18705035971223022
NN_64_6